In [1]:
import datasets
print(datasets.__version__)

c:\Users\venuv\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


4.8.5


In [2]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [3]:
model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1335.30it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [4]:
#apply lora
from peft import LoraConfig, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q", "v"],
    lora_dropout=0.1,
    bias="none",
    task_type="SEQ_2_SEQ_LM"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()

c:\Users\venuv\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\cuda\__init__.py:235: UserWarning: 
NVIDIA GeForce RTX 5050 Laptop GPU with CUDA capability sm_120 is not compatible with the current PyTorch installation.
The current PyTorch install supports CUDA capabilities sm_50 sm_60 sm_61 sm_70 sm_75 sm_80 sm_86 sm_90.
If you want to use the NVIDIA GeForce RTX 5050 Laptop GPU GPU with PyTorch, please check the instructions at https://pytorch.org/get-started/locally/

  warnings.warn(


trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


In [5]:
data = {
    "input": ["What is self attention?"]*100,
    "output": ["Self attention allows tokens to attend to other tokens in a sequence."]*100
}


In [6]:
def preprocess(example):
    inputs = tokenizer(example["input"], padding="max_length", truncation=True, max_length=128)
    labels = tokenizer(example["output"], padding="max_length", truncation=True, max_length=128)

    inputs["labels"] = labels["input_ids"]
    return inputs

In [7]:
from datasets import Dataset
dataset = Dataset.from_dict(data)
print(dataset)

Dataset({
    features: ['input', 'output'],
    num_rows: 100
})


In [8]:
tokenized_dataset = dataset.map(preprocess)

Map: 100%|██████████| 100/100 [00:00<00:00, 859.68 examples/s]


In [9]:
from transformers import Trainer, TrainingArguments

args = TrainingArguments(
    output_dir="./lora_model",
    per_device_train_batch_size=10,
    num_train_epochs=2,
    learning_rate=2e-4,
    use_cpu=True
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=tokenized_dataset,
)

trainer.train()

Step,Training Loss


TrainOutput(global_step=20, training_loss=35.288031005859374, metrics={'train_runtime': 181.6415, 'train_samples_per_second': 1.101, 'train_steps_per_second': 0.11, 'total_flos': 34373763072000.0, 'train_loss': 35.288031005859374, 'epoch': 2.0})

In [11]:
model.print_trainable_parameters()

trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


trainable params: 884,736 || all params: 248,462,592 || trainable%: 0.3561


Step,Training Loss


TrainOutput(global_step=10, training_loss=35.200418090820314, metrics={'train_runtime': 34.2382, 'train_samples_per_second': 0.584, 'train_steps_per_second': 0.292, 'total_flos': 3437376307200.0, 'train_loss': 35.200418090820314, 'epoch': 10.0})

In [14]:
input_text = "what is RAG"
inputs = tokenizer(input_text, return_tensors="pt")
ouputs = model.generate(**inputs, max_new_tokens=100)
print(tokenizer.decode(ouputs[0], skip_special_tokens=True))

a slang term for a group of people who are a part of a group of people


In [15]:
ouputs

tensor([[   0,    3,    9,    3,    7, 4612, 1657,   21,    3,    9,  563,   13,
          151,  113,   33,    3,    9,  294,   13,    3,    9,  563,   13,  151,
            1]])

In [42]:
tokenized_dataset

Dataset({
    features: ['input', 'output', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 2
})